# Transformación de los datos (Punto 5)

En este cuaderno se aplican las transformaciones necesarias sobre el conjunto de datos limpio (`sonrievalencia_citas_limpio.csv`):
1. **Carga del dataset limpio**.
2. **Atributos nuevos con sentido (Feature Engineering)** a partir de fechas, horas y notas de recepción.
3. **Codificación binaria** (Sí = 1, No = 0) y **ordinal** (Urgencia).
4. **Discretización** (tramos de edad) y **codificación One-Hot (Dummies)** de variables categóricas.
5. **Escalado y estandarización** de variables numéricas continuas con `StandardScaler`.
6. **Comprobación y guardado** en `sonrievalencia_citas_transformado.csv`.

## 1. Cargar el dataset limpio

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Cargar el dataset limpio del Punto 4
df = pd.read_csv("sonrievalencia_citas_limpio.csv", sep=";")

print("Dataset limpio cargado correctamente.")
print(f"Dimensiones iniciales: {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head(3)

Dataset limpio cargado correctamente.
Dimensiones iniciales: 2400 filas y 19 columnas.


,id_cita,dni,edad,clinica,cod_clinica,ciudad,tratamiento,urgencia,seguro,primera_visita,visitas_previas,fecha_reserva,fecha_cita,hora_cita,recordatorio_sms,importe_presupuesto_eur,importe_con_iva_eur,notas_recepcion,no_acude
0,T500000,94026698E,56.0,Ruzafa,1,València,Revisión,Baja,Sin seguro,No,2,2024-09-14,2024-10-14,09:45,Sí,29.88,36.15,NaN,No
1,T500001,93583852T,24.0,Ruzafa,1,València,Revisión,Baja,Sin seguro,No,1,2024-09-03,2024-10-10,18:30,Sí,29.47,35.66,Pide factura,No
2,T500002,32677771G,30.0,Ruzafa,1,València,Revisión,Baja,Sin seguro,No,1,2025-02-02,2025-03-06,15:00,Sí,36.38,44.02,NaN,No


## 2. Atributos nuevos con sentido (Feature Engineering)
- **`dias_antelacion`**: diferencia en días entre la fecha de reserva y la fecha de la cita (corrigiendo antelaciones negativas a 0).
- **`dia_semana_cita`**: día de la semana de la cita (0 = Lunes, ..., 6 = Domingo).
- **`es_tarde`**: 1 si la cita es de tarde (>= 14:00), 0 si es de mañana.
- **`miedo_dentista`**: 1 si el paciente menciona miedo en notas de recepción, 0 en caso contrario.
- **`tiene_nota`**: 1 si existen notas de recepción registradas, 0 si está vacío.

In [3]:
# Calcular días de antelación entre reserva y cita
fecha_res = pd.to_datetime(df['fecha_reserva'])
fecha_cit = pd.to_datetime(df['fecha_cita'])

df['dias_antelacion'] = (fecha_cit - fecha_res).dt.days
# Corregir registros con fechas invertidas (antelación negativa -> 0)
df.loc[df['dias_antelacion'] < 0, 'dias_antelacion'] = 0

# Extraer día de la semana de la cita (0 = Lunes ... 6 = Domingo)
df['dia_semana_cita'] = fecha_cit.dt.day_of_week

# Franja horaria: 1 si es de tarde (>= 14:00), 0 si es de mañana
hora_num = df['hora_cita'].str.split(':').str[0].astype(int)
df['es_tarde'] = np.where(hora_num >= 14, 1, 0)

# Aprovechar nulo informativo y patrón de miedo en notas_recepcion (fillna evita convertir NaN en 'nan')
notas_limpias = df['notas_recepcion'].fillna('').str.lower()
df['miedo_dentista'] = np.where(notas_limpias.str.contains('miedo'), 1, 0)
df['tiene_nota'] = np.where(df['notas_recepcion'].notnull(), 1, 0)

print("Nuevos atributos generados con éxito:")
df[['dias_antelacion', 'dia_semana_cita', 'es_tarde', 'miedo_dentista', 'tiene_nota']].head()

Nuevos atributos generados con éxito:


,dias_antelacion,dia_semana_cita,es_tarde,miedo_dentista,tiene_nota
0,30,0,0,0,0
1,37,3,1,0,1
2,32,3,1,0,0
3,44,0,0,0,0
4,16,4,1,0,0


## 3. Codificación binaria y ordinal
- **Binaria**: variables `primera_visita`, `recordatorio_sms` y `no_acude` mapeadas a `Sí: 1, No: 0`.
- **Ordinal**: variable `urgencia` mapeada según jerarquía (`Baja: 1, Media: 2, Alta: 3`).

In [4]:
# Codificación binaria (Sí = 1, No = 0)
mapa_binario = {'Sí': 1, 'No': 0}
for col in ['primera_visita', 'recordatorio_sms', 'no_acude']:
    df[col] = df[col].map(mapa_binario)

# Codificación ordinal (Jerarquía lógica)
mapa_urgencia = {'Baja': 1, 'Media': 2, 'Alta': 3}
df['urgencia'] = df['urgencia'].map(mapa_urgencia)

print("Comprobación de codificación binaria y ordinal:")
print("- primera_visita:", df['primera_visita'].value_counts().to_dict())
print("- recordatorio_sms:", df['recordatorio_sms'].value_counts().to_dict())
print("- no_acude:", df['no_acude'].value_counts().to_dict())
print("- urgencia:", df['urgencia'].value_counts().to_dict())

Comprobación de codificación binaria y ordinal:
- primera_visita: {0: 1931, 1: 469}
- recordatorio_sms: {1: 1522, 0: 878}
- no_acude: {0: 1883, 1: 517}
- urgencia: {1: 1485, 2: 696, 3: 219}


## 4. Discretización y Codificación Dummies (One-Hot Encoding)
- Discretización de `edad` en tramos: `Menor` (0-18], `Joven` (18-35], `Adulto` (35-60], `Senior` (60-120].
- One-Hot Encoding para `clinica`, `seguro`, `tratamiento` y `rango_edad` (conservando columnas originales para el Punto 6).

In [5]:
# Discretizar edad en tramos lógicos
df['rango_edad'] = pd.cut(
    df['edad'], 
    bins=[0, 18, 35, 60, 120], 
    labels=['Menor', 'Joven', 'Adulto', 'Senior'],
    include_lowest=True
)

# Crear columnas dummies (0/1) conservando las columnas originales para el Punto 6
cols_categoricas = ['clinica', 'seguro', 'tratamiento', 'rango_edad']
dummies = pd.get_dummies(df[cols_categoricas], drop_first=True, dtype=int)
df = pd.concat([df, dummies], axis=1)

print(f"Total de dummies generadas: {dummies.shape[1]}")
print("Columnas dummies:", dummies.columns.tolist()[:5], "...")

Total de dummies generadas: 21
Columnas dummies: ['clinica_Campanar', 'clinica_Ruzafa', 'seguro_Privado', 'seguro_Sin seguro', 'tratamiento_Carillas'] ...


## 5. Escalado (Estandarización de variables numéricas continuas)
Estandarización con `StandardScaler` (media = 0, desviación estándar = 1) para las variables numéricas continuas.

In [6]:
scaler = StandardScaler()
cols_escalar = [
    'edad', 
    'visitas_previas', 
    'importe_presupuesto_eur', 
    'importe_con_iva_eur', 
    'dias_antelacion'
]
df[cols_escalar] = scaler.fit_transform(df[cols_escalar])

print("Resumen estadístico de las variables estandarizadas:")
df[cols_escalar].describe().round(2)

Resumen estadístico de las variables estandarizadas:


,edad,visitas_previas,importe_presupuesto_eur,importe_con_iva_eur,dias_antelacion
count,2400.00,2400.00,2400.00,2400.00,2400.00
mean,0.00,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-2.39,-1.45,-0.51,-0.51,-1.64
25%,-0.60,-0.71,-0.48,-0.48,-0.91
50%,0.00,0.02,-0.42,-0.42,0.06
75%,0.60,0.76,-0.16,-0.16,0.85
max,3.26,4.44,5.52,5.52,1.70


## 6. Comprobación y Guardado del dataset transformado

In [7]:
print("Dimensiones tras transformación:", df.shape)
print("Nuevas columnas creadas:", df.shape[1] - 19)

archivo_salida = "sonrievalencia_citas_transformado.csv"
df.to_csv(archivo_salida, sep=";", index=False, encoding="utf-8")
print(f"Dataset transformado guardado exitosamente en: '{archivo_salida}'")

Dimensiones tras transformación: (2400, 46)
Nuevas columnas creadas: 27
Dataset transformado guardado exitosamente en: 'sonrievalencia_citas_transformado.csv'
